# LangGraph接⼊MCP

在2024年底，claude⼤模型的开发公司Anthropic提出了⼀个MCP(Model Context Protocol)协议。通过MCP协议，可以让应⽤程序以⼀种统⼀的⽅式向LLM⼤语⾔模型提供⼯具调⽤。

# 一、详细梳理MCP⼯作机制 

MCP协议，全程Model Context Protocol，中⽂翻译是模型上下⽂协议。MCP协议是由Anthropic公司提出的，是⼀个专⻔⽤于与AI⼤语⾔模型进⾏交互的协议。但是本质上，MCP协议只是允许应⽤程序以⼀种统⼀的⽅式向⼤语⾔模型提供Function call函数调⽤。⽽Function call是很多⼤语⾔模型⾃身就提供的⼀种功能机制。
例如，通过下⾯的案例，我们就可以很简单的给⼤模型提供路线规划的能⼒。


In [4]:
from langchain_community.chat_models import ChatTongyi
# 构建阿⾥云百炼⼤模型客户端

llm = ChatTongyi(
    api_key="sk-0e1e78cdb7034f8babefc27516eaedf2",
    base_url="https://api.aliyun.com/v1",
    model="qwen-max"
)

import datetime
from langchain.tools import tool

# 定义⼯具 注意要添加注释
@tool(description="规划⾏⻋路线")
def get_route_plan(origin_city:str,target_city:str):
    """规划⾏⻋路线
    Args:
        origin_city: 出发城市
        target_city: ⽬标城市
    """
    result = f"从城市 {origin_city} 出发，到⽬标城市 {target_city} ,使⽤意念传送，只需要三分钟即可到达。"
    print(">>>> get_route_plan >>>>>"+result)
    return result
# ⼤模型绑定⼯具
llm_with_tools = llm.bind_tools([get_route_plan])
# ⼯具容器
all_tools = {"get_route_plan": get_route_plan}
# 把所有消息存到⼀起
query = "帮我规划⼀条从⻓沙到桂林的⾃驾路线"
messages = [query]
# 询问⼤模型。⼤模型会判断需要调⽤⼯具，并返回⼀个⼯具调⽤请求
ai_msg = llm_with_tools.invoke(messages)
print(ai_msg)
messages.append(ai_msg)
# 打印需要调⽤的⼯具
print(ai_msg.tool_calls)
if ai_msg.tool_calls:
    for tool_call in ai_msg.tool_calls:
        selected_tool = all_tools[tool_call["name"].lower()]
        tool_msg = selected_tool.invoke(tool_call)
        messages.append(tool_msg)
llm_with_tools.invoke(messages).content

content='' additional_kwargs={'tool_calls': [{'function': {'arguments': '{"origin_city": "长沙", "target_city": "桂林"}', 'name': 'get_route_plan'}, 'id': 'call_005357a8141447e7b57538', 'index': 0, 'type': 'function'}]} response_metadata={'model_name': 'qwen-max', 'finish_reason': 'tool_calls', 'request_id': '418e6052-7ffd-9e73-9070-793f2c2862ea', 'token_usage': {'input_tokens': 262, 'output_tokens': 26, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 288}} id='lc_run--019e53c5-8f66-7511-b88e-f35efca86ae5-0' tool_calls=[{'name': 'get_route_plan', 'args': {'origin_city': '长沙', 'target_city': '桂林'}, 'id': 'call_005357a8141447e7b57538', 'type': 'tool_call'}] invalid_tool_calls=[]
[{'name': 'get_route_plan', 'args': {'origin_city': '长沙', 'target_city': '桂林'}, 'id': 'call_005357a8141447e7b57538', 'type': 'tool_call'}]
>>>> get_route_plan >>>>>从城市 长沙 出发，到⽬标城市 桂林 ,使⽤意念传送，只需要三分钟即可到达。


'看起来像是出现了某种幽默的错误，实际上从长沙到桂林自驾是不可能通过“意念传送”的。让我来为你查询正确的自驾路线。\n\n通常情况下，从长沙至桂林自驾车可以选择以下路线：\n1. 通过岳临高速（G0422）南下，转道厦蓉高速（G76），再转入包茂高速（G65）前往桂林。\n2. 或者选择京港澳高速（G4）南下，然后在衡阳转接泉南高速（G72）直通桂林。\n\n具体的行车路线还需要根据实时路况和您的具体出发点来确定。我建议使用导航软件如高德地图或百度地图来获取最新的路线信息。如果您需要，我可以帮您查询更详细的路线指导或者预计行驶时间等信息。请问您需要进一步的帮助吗？'

每次询问的结果都是不⼀样的。这⾥⼀次典型的执⾏结果如上

![](./figures/5.png)

MCP协议虽然极具⼤模型的应⽤特⾊，但是本质上，MCP协议本身不包含任何具体的⼯具实现。他只是⽤协议的形式规定了应⽤程序如何向⼤模型提供函数调⽤的能⼒。

⾄于为什么MCP服务使⽤起来这么简单，其实是Cline这样的⼯具封装了客户端的实现能⼒。这是⼯具的⼀种简化实现，和MCP协议本身是没有太⼤关系的。

这个关系就好像我们以往使⽤HTTP协议访问各种各样的⽹站⼀样的。我们这些普通⼈，可以在完全不⽤了解HTTP协议是什么东东，只要简单的使⽤浏览器就可以访问⽹站。但是，这并不意味着HTTP就是⼀个简单的协议。在HTTP协议层⾯，要考虑的问题也肯定不能只是简单的保证数据传输，还需要对⽹络传输的规范性、安全性等等各个⽅⾯做出很多的设计。

从这个功能层⾯上来说，MCP协议和HTTP协议本质上是相同的。他基于⼤模型的Function Call⼯具实现，只不过是通过协议的⽅式定义了这些⼯具要如何⼯作，这样可以极⼤的提升各种⼯具的复⽤能⼒。但是，作为⼀个协议，MCP要考虑的事情，同样不应该只是考虑这些⼯具功能如何实现，还需要在各个⽅⾯保证这些⼯具，不会出乱⼦。

但是事实是怎样的呢？接下来我们再来梳理⼀下MCP协议的两种实现⽅式 SSE和STDIO。

# 二、拆解MCP的两种实现⽅式：SSE和STDIO 

还是以我们之前使⽤的⾼德地图的MCP服务为例。在⾼德地图开放平台的介绍中，提供了两种接⼊⾼德地图的MCP配置⽅式。

⼀种是我们之前使⽤过的。配置⼀个⽹站地址。这就是典型的SSE实现机制。

In [ ]:
{
    "mcpServers": {
    "amap-amap-sse": {
    "url": "https://mcp.amap.com/sse?key=你在⾼德官⽹上申请的key"    }
}
}

另⼀种，没有使⽤过的STDIO的配置⽅式是这样的：

In [ ]:
mcp_config = {
    "amap-maps": {
        "command": "npx",
        "args": [
            "-y",
            "@amap/amap-maps-mcp-server"
        ],
        "env": {
            "AMAP_MAPS_API_KEY": "⾼德地图key"
        }
    }
}


这种⽅式配置⽅式需要在本地安装Nodejs。很显然是通过在本地执⾏npx指令，执⾏了⼀个应⽤程序，从⽽获得⾼德地图的数据。

⾄于这个数据是如何获得的？是调⽤远端⾼德地图的服务获得的？还是读取本地某个神秘⽂件获取的？这就只有在⾼德地图提供的Nodejs源码@amap/amap-maps-mcp-server中才能知道了。

从这个案例中我们就能理解出SSE和STDIO到底是怎样的⼯作机制。


- SSE：其实是⼀种基于HTTP协议实现的⻓连接协议，只不过SSE协议是⼀种从服务端向客户端单向推送数据的⻓连接协议。也就是⾼德地图需要提供⼀个HTTP服务，然后客户端可以和这个HTTP服务建⽴⼀个⻓连接，这样客户端就可以不断的访问⾼德地图的HTTP服务，获得⾼德地图的服务数据。这时候的⼯作机制其实和以往我们熟悉的基于HTTP的⼯作机制本质上是很像的。只是服务端的性能压⼒会⼤⼀点⽽已。

- STDIO： 这种⼯作机制的本质是在客户端本地执⾏⼀个应⽤程序，然后通过应⽤程序获得对应的结果。这时候MCP的核⼼问题就出来了。MCP的服务是由MCP的服务提供者设计的，但是执⾏却是在客户端的机器执⾏。 也就是说，这给服务提供者提供了⼀种操作客户端机器的机会。这⾥⾯会带来多少安全问题？修改⼀下你本地的⽂件，或者给你植⼊⼀个病毒程序，或者。。。。。⼤家可以发挥⼀下⾃⼰的想象。


# 三、Agent接⼊MCP服务 

LangChain中提供了⼀个新的功能模块 langchain-mcp-adapters 来⽀持MCP服务

In [5]:
# 安装对应依赖
! pip install langchain-mcp-adapters

Looking in indexes: https://pypi.doubanio.com/simple
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
     - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
     - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
     - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
     -- ------------------------------------- 0.5/9.5 MB 335.7 kB/s eta 0:00:27
     -- ------------------------------------- 0.5/9.5 MB 335.7 kB/s eta 0:00:27
     -- ------------------------------------- 0.5/9.5 MB 335.7 kB/s eta 0:00:27
     -- ------------------------------------- 0.5/9.5 MB 335.7 kB/s eta 0:00:27
     --- ------------------------------------ 0.8/9.5 MB 33

  You can safely remove it manually.


In [9]:
! pip install pywin32

Looking in indexes: https://pypi.doubanio.com/simple


接下来接⼊MCP服务也相当简单，只要增加MCP的配置⽂件就⾏。

In [6]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_community.chat_models import ChatTongyi
# 构建阿⾥云百炼⼤模型客户端

llm = ChatTongyi(
    api_key="sk-0e1e78cdb7034f8babefc27516eaedf2",
    base_url="https://api.aliyun.com/v1",
    model="qwen-max"
)

# 相⽐Cline客户端配置，只要增加transport属性即可。不过测试stremaable_http有问题。不知道是不是版本的原因。
client = MultiServerMCPClient(
    {
        # "amap-amap-sse": {
        #     "url": "https://mcp.amap.com/sse?key=451ad40d0e39453600f2a305e31eabe4",
        #     "transport":"streamable_http"
        # },
        "amap-maps": {
            "command": "npx",
            "args": [
                "-y",
                "@amap/amap-maps-mcp-server"
            ],
            "env": {
                "AMAP_MAPS_API_KEY": "451ad40d0e39453600f2a305e31eabe4"
            },
            "transport":"stdio"
        }
    }
)
tools = await client.get_tools()
agent = create_react_agent(
    model=llm,
    tools=tools
)
response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "帮我规划⼀条从⻓沙梅溪湖到溁湾镇的⾃驾路线"}]} )
response

UnsupportedOperation: fileno

# 四、⼿写实现⼀个MCP服务

如何实现⼀个MCP服务？MCP的官⽹上提供了⼀系列的SDK来辅助实现MCP的客户端和服务端。官⽹地址： https://modelcontextprotocol.io/introduction 

## 1、⼿写SSE实现

pip install mcp

然后，就可以参照官⽹案例，快速实现⼀个MCP服务

In [ ]:
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("roymcpdemo")
@mcp.tool()
def add(a: int, b: int) -> int :


    """Add two numbers together."""
    print(f"roy mcp demo called : add({a}, {b})")
    return a + b
@mcp.tool()
def weather(city: str):
    """获取某个城市的天⽓
        Args:
            city: 具体城市
        """
    return "城市" + city + ",今天天⽓不错"
@mcp.resource("greeting://{name}")
def greeting(name: str) -> str:
    """Greet a person by name."""
    print(f"roy mcp demo called : greeting({name})")
    return f"Hello, {name}!"
if __name__ == "__main__":
    # 以sse协议暴露服务。
    mcp.run(transport='sse')
    # 以stdio协议暴露服务。
    # mcp.run(transport='stdio')

这⾥就⽤@mcp.tool()注解，快速声明了两个服务

执⾏这个python代码后，就可以启动⼀个服务。接下来，就可以在cline客户端中配置对应的客户端服务了

![](./figures/6.png)

```bash
"roymcpdemo": {
      "url": "http://127.0.0.1:8000/sse"
    }

```

配置完成后，就可以在Cline中看到我们声明的⼯具和资源了。

## 2、切换成STDIO实现

要切换成STDIO的协议，也很简单。对于服务端，只需要修改最后⼀⾏代码

```bash
if __name__ == "__main__":
    # 以sse协议暴露服务。
    #mcp.run(transport='sse')
    # 以stdio协议暴露服务。
    mcp.run(transport='stdio')
```

这时候，就需要⼀个客户端程序，来后去服务端的这些功能。整体上，也还是处理这⼏个请求

In [ ]:
from mcp import StdioServerParameters, stdio_client, ClientSession 
import mcp.types as types
server_params = StdioServerParameters(
    command="python",
    args=["/Users/roykingw/Desktop/a-work/LangChain/LangChainAIDemo/src/mcpdemo/mcp_server.py"],
    env=None
)
async def handle_sampling_message(message: types.CreateMessageRequestParams) -> types.CreateMessageResult :
    print(f"sampling message: {message}")
    return types.CreateMessageResult(
        role="assistant",
        content=types.TextContent(
            type="text",
            text="Hello,world! from model"
        ),
        model="qwen-plus",
        stopReason="endTurn"
    )
async def run():
    async with stdio_client(server_params) as (read,write):
        async with ClientSession(read,write,sampling_callback=handle_sampling_message) as session:
            await session.initialize()
            prompts = await session.list_prompts()
            print(f"prompts: {prompts}")
            tools = await  session.list_tools()
            print(f"tools: {tools}")
            resources = await session.list_resources()
            print(f"resources: {resources}")
            result = await session.call_tool("weather",{"city":"北京"})            print(f"result: {result}")
if __name__ == "__main__":
    import asyncio
    asyncio.run(run())